# 🔌 Notebook 4 — Inference Engine & Backend Integration
## AuthentiQ – Wiring ML Model → FastAPI

This notebook:
1. Defines the `AuthentiQDetector` inference class
2. Tests prediction on custom text samples
3. Generates sentence-level heatmap JSON (matching the API schema)
4. Shows exactly how the FastAPI `analysis.py` endpoint should call the model
5. Writes `../ml/detector.py` — the serving module

In [1]:
import joblib
import json
import re
import numpy as np
import warnings
from pathlib import Path
from scipy.sparse import hstack, csr_matrix
from dataclasses import dataclass, asdict
from typing import List, Tuple, Optional

warnings.filterwarnings('ignore')
MODELS_DIR = Path('../models')
print('✅ Ready')

✅ Ready


## 1. Inference Class Definition

In [2]:
@dataclass
class SentenceSpan:
    text:  str
    heat:  str   # 'low' | 'mid' | 'high'
    score: float
    start: int
    end:   int

@dataclass
class DetectionResult:
    ai_probability:     float
    overall_score:      float   # 0-100
    verdict:            str
    confidence_level:   str
    model_version:      str
    word_count:         int
    char_count:         int
    sentence_spans:     List[SentenceSpan]
    dimensions:         dict


class AuthentiQDetector:
    """Singleton inference engine for AI-text detection."""
    
    def __init__(self, models_dir: Path):
        self.models_dir = Path(models_dir)
        self._load()
    
    def _load(self):
        print('Loading model artifacts...')
        self.tfidf_char = joblib.load(self.models_dir / 'tfidf_char.pkl')
        self.tfidf_word = joblib.load(self.models_dir / 'tfidf_word.pkl')
        self.sgd_clf    = joblib.load(self.models_dir / 'sgd_classifier.pkl')
        self.lgb_model  = joblib.load(self.models_dir / 'lgb_classifier.pkl')
        with open(self.models_dir / 'model_meta.json') as f:
            self.meta = json.load(f)
        self.threshold  = self.meta['optimal_threshold']
        self.w_sgd      = self.meta['sgd_weight']
        self.w_lgb      = self.meta['lgb_weight']
        print(f'✅ Models loaded | Version: {self.meta["model_version"]} | AUC: {self.meta["ensemble_auc"]}')
    
    @staticmethod
    def _preprocess(text: str) -> str:
        text = str(text).strip()
        text = re.sub(r'\r\n|\r', '\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        return text.lower()
    
    @staticmethod
    def _linguistic_features(texts) -> np.ndarray:
        rows = []
        for text in texts:
            words = text.split()
            sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
            sl    = [len(s.split()) for s in sents]
            rows.append([
                len(text), len(words), len(sents),
                len(set(words))/max(len(words),1),
                np.std(sl) if len(sl)>1 else 0, np.mean(sl) if sl else 0,
                text.count(',')/max(len(words),1),
                text.count('!')/max(len(words),1),
                text.count('\n\n')+1,
                text.count('furthermore')+text.count('additionally'),
                text.count('moreover')+text.count('delve'),
                sum(1 for w in words if w.isupper() and len(w)>1)/max(len(words),1),
            ])
        return np.array(rows, dtype=np.float32)
    
    def _build_features(self, texts):
        X_char = self.tfidf_char.transform(texts)
        X_word = self.tfidf_word.transform(texts)
        X_ling = self._linguistic_features(texts)
        return hstack([X_char, X_word, csr_matrix(X_ling)])
    
    def _ensemble_proba(self, X) -> np.ndarray:
        p_sgd = self.sgd_clf.predict_proba(X)[:, 1]
        p_lgb = self.lgb_model.predict_proba(X)[:, 1]
        return self.w_sgd*p_sgd + self.w_lgb*p_lgb
    
    @staticmethod
    def _heat_label(score: float) -> str:
        if score >= 0.70:  return 'high'
        if score >= 0.40:  return 'mid'
        return 'low'
    
    @staticmethod
    def _verdict(prob: float, threshold: float) -> Tuple[str, str]:
        if prob >= 0.85:   return 'Highly Likely AI',    'High Confidence'
        if prob >= threshold: return 'Likely AI',        'Medium Confidence'
        if prob >= 0.35:   return 'Uncertain / Mixed',   'Low Confidence'
        return 'Likely Human',                           'Medium Confidence'
    
    def detect(self, text: str) -> DetectionResult:
        """Run full detection pipeline on a single text."""
        clean = self._preprocess(text)
        
        # --- Global score ---
        X_global  = self._build_features([clean])
        ai_prob   = float(self._ensemble_proba(X_global)[0])
        
        # --- Sentence-level spans ---
        raw_sentences = re.split(r'(?<=[.!?])\s+', text)
        raw_sentences = [s.strip() for s in raw_sentences if len(s.strip()) > 10]
        
        sent_spans = []
        if raw_sentences:
            clean_sents = [self._preprocess(s) for s in raw_sentences]
            X_sents     = self._build_features(clean_sents)
            sent_scores = self._ensemble_proba(X_sents)
            
            cursor = 0
            for sent, score in zip(raw_sentences, sent_scores):
                start = text.find(sent, cursor)
                end   = start + len(sent) if start >= 0 else cursor + len(sent)
                sent_spans.append(SentenceSpan(
                    text=sent[:200],
                    heat=self._heat_label(float(score)),
                    score=round(float(score), 3),
                    start=max(start, 0),
                    end=end,
                ))
                cursor = end
        
        # --- Dimension breakdown ---
        words = clean.split()
        sents_c = [s.strip() for s in re.split(r'[.!?]+', clean) if s.strip()]
        sl = [len(s.split()) for s in sents_c]
        
        dims = {
            'AI-Typical Phrasing':  round(min(100, (clean.count('furthermore') + clean.count('additionally') + clean.count('delve') + clean.count('moreover')) * 12 + ai_prob*40), 1),
            'Structural Patterns':  round(min(100, max(0, (1 - (np.std(sl)/max(np.mean(sl),1) if sl else 0)) * 100 * 0.6 + ai_prob*40)), 1),
            'Source Similarity':    round(ai_prob * 50, 1),
            'Style Consistency':    round(min(100, (1 - len(set(words))/max(len(words),1)) * 100 * 0.5 + ai_prob*50), 1),
        }
        
        verdict, confidence = self._verdict(ai_prob, self.threshold)
        
        return DetectionResult(
            ai_probability=round(ai_prob, 4),
            overall_score=round(ai_prob * 100, 1),
            verdict=verdict,
            confidence_level=confidence,
            model_version=self.meta['model_version'],
            word_count=len(text.split()),
            char_count=len(text),
            sentence_spans=sent_spans,
            dimensions=dims,
        )

print('✅ AuthentiQDetector class defined')

✅ AuthentiQDetector class defined


## 2. Load & Test Inference

In [3]:
detector = AuthentiQDetector(MODELS_DIR)

Loading model artifacts...
✅ Models loaded | Version: authentiq-v1.0 | AUC: 0.997


In [4]:
# ── Test 1: Typical AI Text ─────────────────────────────────────────────────
ai_text = """
In today's rapidly evolving technological landscape, artificial intelligence has fundamentally 
transformed the way organizations approach problem-solving and decision-making. Furthermore, 
the integration of machine learning algorithms has enabled businesses to leverage vast amounts 
of data to drive actionable insights. Moreover, these advancements have led to significant 
improvements in operational efficiency and competitive advantage. Additionally, it is crucial 
to delve deeper into the ethical implications of these technologies to ensure responsible 
implementation. Overall, the potential of AI to reshape industries cannot be overstated.
"""

result = detector.detect(ai_text)
print('=== AI Text Test ===')
print(f'  AI Probability:   {result.ai_probability:.4f}  ({result.overall_score:.1f}%)')
print(f'  Verdict:          {result.verdict}')
print(f'  Confidence:       {result.confidence_level}')
print(f'  Word count:       {result.word_count}')
print(f'  Dimensions:       {result.dimensions}')
print(f'  Spans ({len(result.sentence_spans)}):')
for span in result.sentence_spans:
    print(f'    [{span.heat.upper():4}] ({span.score:.3f}) {span.text[:70]}...')

=== AI Text Test ===
  AI Probability:   0.9704  (97.0%)
  Verdict:          Highly Likely AI
  Confidence:       High Confidence
  Word count:       81
  Dimensions:       {'AI-Typical Phrasing': 86.8, 'Structural Patterns': np.float64(86.8), 'Source Similarity': 48.5, 'Style Consistency': 57.8}
  Spans (5):
    [HIGH] (0.962) In today's rapidly evolving technological landscape, artificial intell...
    [HIGH] (0.961) Furthermore, 
the integration of machine learning algorithms has enabl...
    [HIGH] (0.935) Moreover, these advancements have led to significant 
improvements in ...
    [HIGH] (0.955) Additionally, it is crucial 
to delve deeper into the ethical implicat...
    [HIGH] (0.903) Overall, the potential of AI to reshape industries cannot be overstate...


In [5]:
# ── Test 2: Typical Human Text ──────────────────────────────────────────────
human_text = """
I never thought I'd be writing something like this but here we are. Last summer I spent three 
weeks backpacking across rural Portugal, mostly alone, mostly confused, and honestly it was one 
of the best things I've done. You meet the weirdest people at hostels. There was this guy from 
Belgium who insisted on showing everyone card tricks that never worked. Every single time he'd 
go "wait, wait" and do it again. But you know what? We laughed so hard. Anyway, I'm getting 
off track. The point is—travel messes with your sense of time in a way that's kinda hard to 
explain. One afternoon you're eating pastel de nata by a fountain and suddenly you're not 
anxious about any of the stuff that seemed huge back home.
"""

result2 = detector.detect(human_text)
print('=== Human Text Test ===')
print(f'  AI Probability:   {result2.ai_probability:.4f}  ({result2.overall_score:.1f}%)')
print(f'  Verdict:          {result2.verdict}')
print(f'  Confidence:       {result2.confidence_level}')

=== Human Text Test ===
  AI Probability:   0.6842  (68.4%)
  Verdict:          Likely AI
  Confidence:       Medium Confidence


## 3. Export Inference Module `detector.py`

In [6]:
DETECTOR_CODE = '''
"""
AuthentiQ ML Inference Engine
detector.py — loads trained TF-IDF + ensemble models and exposes detect() for FastAPI.
"""
import joblib
import json
import re
import numpy as np
from pathlib import Path
from scipy.sparse import hstack, csr_matrix
from dataclasses import dataclass
from typing import List, Tuple

MODELS_DIR = Path(__file__).parent / "models"


@dataclass
class SentenceSpan:
    text:  str
    heat:  str
    score: float
    start: int
    end:   int


@dataclass
class DetectionResult:
    ai_probability:   float
    overall_score:    float
    verdict:          str
    confidence_level: str
    model_version:    str
    word_count:       int
    char_count:       int
    sentence_spans:   List[SentenceSpan]
    dimensions:       dict


class AuthentiQDetector:
    """Singleton inference engine. Load once at startup via get_detector()."""

    def __init__(self, models_dir: Path = MODELS_DIR):
        self.models_dir = Path(models_dir)
        self.tfidf_char = joblib.load(self.models_dir / "tfidf_char.pkl")
        self.tfidf_word = joblib.load(self.models_dir / "tfidf_word.pkl")
        self.sgd_clf    = joblib.load(self.models_dir / "sgd_classifier.pkl")
        self.lgb_model  = joblib.load(self.models_dir / "lgb_classifier.pkl")
        with open(self.models_dir / "model_meta.json") as f:
            self.meta = json.load(f)
        self.threshold = self.meta["optimal_threshold"]
        self.w_sgd     = self.meta["sgd_weight"]
        self.w_lgb     = self.meta["lgb_weight"]

    @staticmethod
    def _preprocess(text: str) -> str:
        text = str(text).strip()
        text = re.sub(r"\r\n|\r", "\\n", text)
        text = re.sub(r"[ \\t]{2,}", " ", text)
        return text.lower()

    @staticmethod
    def _linguistic_features(texts) -> np.ndarray:
        rows = []
        for text in texts:
            words = text.split()
            sents = [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]
            sl    = [len(s.split()) for s in sents]
            rows.append([
                len(text), len(words), len(sents),
                len(set(words)) / max(len(words), 1),
                np.std(sl) if len(sl) > 1 else 0,
                np.mean(sl) if sl else 0,
                text.count(",") / max(len(words), 1),
                text.count("!") / max(len(words), 1),
                text.count("\\n\\n") + 1,
                text.count("furthermore") + text.count("additionally"),
                text.count("moreover") + text.count("delve"),
                sum(1 for w in words if w.isupper() and len(w) > 1) / max(len(words), 1),
            ])
        return np.array(rows, dtype=np.float32)

    def _build_features(self, texts):
        X_char = self.tfidf_char.transform(texts)
        X_word = self.tfidf_word.transform(texts)
        X_ling = self._linguistic_features(texts)
        return hstack([X_char, X_word, csr_matrix(X_ling)])

    def _ensemble_proba(self, X) -> np.ndarray:
        p_sgd = self.sgd_clf.predict_proba(X)[:, 1]
        p_lgb = self.lgb_model.predict_proba(X)[:, 1]
        return self.w_sgd * p_sgd + self.w_lgb * p_lgb

    @staticmethod
    def _heat_label(score: float) -> str:
        if score >= 0.70: return "high"
        if score >= 0.40: return "mid"
        return "low"

    @staticmethod
    def _verdict(prob: float, threshold: float) -> Tuple[str, str]:
        if prob >= 0.85:      return "Highly Likely AI",  "High Confidence"
        if prob >= threshold: return "Likely AI",         "Medium Confidence"
        if prob >= 0.35:      return "Uncertain / Mixed", "Low Confidence"
        return "Likely Human",                            "Medium Confidence"

    def detect(self, text: str) -> DetectionResult:
        clean    = self._preprocess(text)
        X_global = self._build_features([clean])
        ai_prob  = float(self._ensemble_proba(X_global)[0])

        raw_sentences = [s.strip() for s in re.split(r"(?<=[.!?])\\s+", text) if len(s.strip()) > 10]
        sent_spans    = []
        if raw_sentences:
            clean_sents = [self._preprocess(s) for s in raw_sentences]
            X_sents     = self._build_features(clean_sents)
            sent_scores = self._ensemble_proba(X_sents)
            cursor = 0
            for sent, score in zip(raw_sentences, sent_scores):
                start = text.find(sent, cursor)
                end   = start + len(sent) if start >= 0 else cursor + len(sent)
                sent_spans.append(SentenceSpan(
                    text=sent[:200], heat=self._heat_label(float(score)),
                    score=round(float(score), 3), start=max(start, 0), end=end,
                ))
                cursor = end

        words   = clean.split()
        sents_c = [s.strip() for s in re.split(r"[.!?]+", clean) if s.strip()]
        sl      = [len(s.split()) for s in sents_c]
        dims = {
            "AI-Typical Phrasing":  round(min(100, (clean.count("furthermore") + clean.count("additionally") + clean.count("delve") + clean.count("moreover")) * 12 + ai_prob * 40), 1),
            "Structural Patterns":  round(min(100, max(0, (1 - (np.std(sl) / max(np.mean(sl), 1) if sl else 0)) * 100 * 0.6 + ai_prob * 40)), 1),
            "Source Similarity":    round(ai_prob * 50, 1),
            "Style Consistency":    round(min(100, (1 - len(set(words)) / max(len(words), 1)) * 100 * 0.5 + ai_prob * 50), 1),
        }
        verdict, confidence = self._verdict(ai_prob, self.threshold)
        return DetectionResult(
            ai_probability=round(ai_prob, 4),
            overall_score=round(ai_prob * 100, 1),
            verdict=verdict,
            confidence_level=confidence,
            model_version=self.meta["model_version"],
            word_count=len(text.split()),
            char_count=len(text),
            sentence_spans=sent_spans,
            dimensions=dims,
        )


# Module-level singleton — lazy loaded
_detector_instance: AuthentiQDetector | None = None

def get_detector() -> AuthentiQDetector:
    global _detector_instance
    if _detector_instance is None:
        _detector_instance = AuthentiQDetector()
    return _detector_instance
'''

output_path = Path('../detector.py')
output_path.write_text(DETECTOR_CODE.strip())
print(f'✅ Exported to {output_path.resolve()}')

✅ Exported to /Users/sanyamaggarwal/Developer/Hactoberfest/AuthentiQ/AI-Content-Intelligence-Platform/ai/ml/detector.py


## 4. How to Wire into FastAPI `analysis.py`

In [7]:
# This is a READ-ONLY illustration of the diff needed in Backend/app/api/v1/endpoints/analysis.py
diff_code = '''
# ─── BEFORE (heuristic stubs) ──────────────────────────────────────────
# overall_score = 74.0
# ai_prob = 0.74
# verdict = "Likely AI"

# ─── AFTER (real ML model) ─────────────────────────────────────────────
from ai.ml.detector import get_detector  # module path from project root

@router.post("/text", response_model=APIResponse[TextAnalysisResponse])
async def run_text_analysis(req: TextAnalysisRequest, ...) -> APIResponse[TextAnalysisResponse]:
    detector = get_detector()           # lazy singleton — loads once
    result   = detector.detect(req.text)
    
    sentence_spans = [
        SentenceSpan(text=s.text, heat=s.heat, start=s.start, end=s.end)
        for s in result.sentence_spans
    ]
    dimensions = [
        DimensionScore(name=k, pct=v, description="")
        for k, v in result.dimensions.items()
    ]
    analysis_res = TextAnalysisResponse(
        id=f"tx_{uuid.uuid4().hex[:8]}",
        overall_score=result.overall_score,
        ai_probability=result.ai_probability,
        confidence_level=result.confidence_level,
        verdict=result.verdict,
        word_count=result.word_count,
        char_count=result.char_count,
        model_version=result.model_version,
        sentence_spans=sentence_spans,
        dimensions=dimensions,
        caveat="Results are probabilistic. Not proof of authorship.",
        analyzed_at=datetime.now(timezone.utc),
    )
    return APIResponse(data=analysis_res, message="Text analysis completed")
'''
print(diff_code)


# ─── BEFORE (heuristic stubs) ──────────────────────────────────────────
# overall_score = 74.0
# ai_prob = 0.74
# verdict = "Likely AI"

# ─── AFTER (real ML model) ─────────────────────────────────────────────
from ai.ml.detector import get_detector  # module path from project root

@router.post("/text", response_model=APIResponse[TextAnalysisResponse])
async def run_text_analysis(req: TextAnalysisRequest, ...) -> APIResponse[TextAnalysisResponse]:
    detector = get_detector()           # lazy singleton — loads once
    result   = detector.detect(req.text)

    sentence_spans = [
        SentenceSpan(text=s.text, heat=s.heat, start=s.start, end=s.end)
        for s in result.sentence_spans
    ]
    dimensions = [
        DimensionScore(name=k, pct=v, description="")
        for k, v in result.dimensions.items()
    ]
    analysis_res = TextAnalysisResponse(
        id=f"tx_{uuid.uuid4().hex[:8]}",
        overall_score=result.overall_score,
        ai_probability=result.ai_probab

## ✅ Pipeline Complete!

```
ai/
└── ml/
    ├── notebooks/
    │   ├── 01_data_exploration.ipynb     ← EDA
    │   ├── 02_model_training.ipynb       ← Train SGD + LightGBM ensemble
    │   ├── 03_evaluation.ipynb           ← ROC, CM, per-source, heatmap
    │   └── 04_inference_integration.ipynb ← This notebook
    ├── models/
    │   ├── tfidf_char.pkl
    │   ├── tfidf_word.pkl
    │   ├── sgd_classifier.pkl
    │   ├── lgb_classifier.pkl
    │   └── model_meta.json
    └── detector.py                        ← Import this in FastAPI
```

**Run notebooks in order:**
```bash
cd ai/ml/notebooks
jupyter lab
# 01 → 02 → 03 → 04
```